# PoC — O ranking de reclamacoes do BC move o preco do Itau?

**Tese (versao curta):** nao e *quantas* reclamacoes o banco tem, e *quao graves* elas sao — e o mercado reage a isso quando o BC divulga o ranking.

Este notebook responde o **pre-requisito** dessa tese: o sinal (a divulgacao) move o preco? Testamos no **Itau (ITUB4)**.

Metodo: **event study**. Para cada divulgacao, medimos o quanto a acao se moveu *alem do que o mercado explicaria* — o **retorno anormal**. Se o ranking nao carrega informacao, isso deveria ser ruido em torno de zero.

> Rode as celulas em ordem (Ctrl+F9 = Run all). Tudo e auto-contido: os precos sao baixados na hora.

In [ ]:
# 1) Dependencias (yfinance baixa precos; scipy faz os testes estatisticos)
!pip -q install yfinance scipy

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 4.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

## 2) As datas dos eventos

Datas **oficiais** de divulgacao do ranking de Bancos, copiadas do calendario do BC (escopo 2022+, onde as datas sao exatas). 2022 Q2 nao existe (transicao de metodologia do BC). O horario e ~14h30/15h: mercado aberto, entao a reacao cabe no dia 0 e transborda pro dia +1.

In [ ]:
import datetime as dt

DATAS_DIVULGACAO = {
    (2022, 1): dt.date(2022, 7, 21),
    (2022, 3): dt.date(2022, 10, 20),
    (2022, 4): dt.date(2023, 1, 19),
    (2023, 1): dt.date(2023, 4, 20),
    (2023, 2): dt.date(2023, 7, 25),
    (2023, 3): dt.date(2023, 10, 31),
    (2023, 4): dt.date(2024, 1, 25),
    (2024, 1): dt.date(2024, 4, 25),
    (2024, 2): dt.date(2024, 7, 30),
    (2024, 3): dt.date(2024, 10, 24),
    (2024, 4): dt.date(2025, 1, 23),
    (2025, 1): dt.date(2025, 4, 24),
    (2025, 2): dt.date(2025, 7, 24),
    (2025, 3): dt.date(2025, 10, 23),
    (2025, 4): dt.date(2026, 1, 22),
}
print(f'{len(DATAS_DIVULGACAO)} eventos')

## 3) Precos e retornos

Baixamos o **fechamento ajustado** (por dividendos/splits) do Itau e do Ibovespa (o 'mercado'). O retorno diario e `R_t = P_t / P_{t-1} - 1`.

In [ ]:
px = yf.download(['ITUB4.SA', '^BVSP'], start='2021-06-01', auto_adjust=True, progress=False)['Close']
px = px.dropna()
ret = px.pct_change().dropna()
ret.columns = ['r_acao', 'r_mkt']  # ITUB4.SA, ^BVSP (ordem alfabetica do yfinance)
print(f'{len(ret)} pregoes, de {ret.index.min().date()} a {ret.index.max().date()}')
ret.tail()

## 4) O motor do event study

Para cada evento:
1. **Janela de estimacao** (120 pregoes antes, de -130 a -11): aprende o 'normal' do Itau via **modelo de mercado** `R_acao = alpha + beta * R_mercado` (regressao OLS).
2. **Janela do evento** (-5 a +5): calcula o **retorno anormal** `AR = R_real - (alpha + beta * R_mercado)`.
3. **CAR** = soma dos AR numa janela (ex.: dias 0 e +1).

O dia 0 e o 1o pregao na data de divulgacao (ou seguinte, se feriado).

In [ ]:
EST_INI, EST_FIM = -130, -11   # estimacao: 120 pregoes
EVT_INI, EVT_FIM = -5, +5      # janela do evento
JANELAS = {'CAR(0,0)': (0, 0), 'CAR(0,1)': (0, 1), 'CAR(-1,1)': (-1, 1), 'CAR(-5,5)': (-5, 5)}

def event_study(ret, datas):
    dias = ret.index
    linhas, ar_por_evento = [], {}
    for (ano, q), data in sorted(datas.items()):
        t0 = dias.searchsorted(pd.Timestamp(data))
        if t0 + EST_INI < 0 or t0 + EVT_FIM >= len(dias):
            continue  # historico insuficiente nas pontas
        est = ret.iloc[t0 + EST_INI : t0 + EST_FIM + 1]
        beta, alpha = np.polyfit(est['r_mkt'], est['r_acao'], 1)
        evt = ret.iloc[t0 + EVT_INI : t0 + EVT_FIM + 1]
        ar = evt['r_acao'] - (alpha + beta * evt['r_mkt'])
        ar.index = range(EVT_INI, EVT_FIM + 1)  # dia relativo
        ar_por_evento[f'{ano} Q{q}'] = ar
        linha = {'evento': f'{ano} Q{q}', 'dia0': dias[t0].date(),
                 'alpha': alpha, 'beta': beta}
        for nome, (a, b) in JANELAS.items():
            linha[nome] = ar.loc[a:b].sum()
        linhas.append(linha)
    return pd.DataFrame(linhas), pd.DataFrame(ar_por_evento)

car, ar_mat = event_study(ret, DATAS_DIVULGACAO)
print(f'{len(car)} eventos processados')

## 5) Os 15 resultados, um por evento

`CAR(0,1)` = reacao anormal no dia da divulgacao + dia seguinte. **Negativo = a acao caiu mais do que o mercado.**

In [ ]:
tabela = car[['evento', 'dia0', 'beta', 'CAR(0,1)']].copy()
tabela['CAR(0,1) %'] = (tabela['CAR(0,1)'] * 100).round(2)
tabela['reacao'] = np.where(tabela['CAR(0,1)'] < 0, 'queda', 'alta')
tabela[['evento', 'dia0', 'beta', 'CAR(0,1) %', 'reacao']].round(2)

In [ ]:
# grafico: reacao por evento (vermelho = queda, azul = alta)
v = car['CAR(0,1)'] * 100
cores = ['#e34948' if x < 0 else '#2a78d6' for x in v]
plt.figure()
plt.bar(car['evento'], v, color=cores)
plt.axhline(0, color='#444', linewidth=0.8)
plt.ylabel('reacao anormal CAR(0,+1)  (%)')
plt.title('Reacao do Itau a cada divulgacao do ranking do BC')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

## 6) Os testes: e sinal ou e acaso?

- **Teste t**: a media dos CARs e diferente de zero?
- **Teste do sinal** (nao-parametrico): quantos eventos sao negativos? Sob 'sem efeito', ~50%. Nao assume normalidade — importante com so 15 eventos.

In [ ]:
print('Resultado consolidado (15 eventos):\n')
for nome in JANELAS:
    x = car[nome].values
    t, p_t = stats.ttest_1samp(x, 0.0)
    neg = int((x < 0).sum())
    p_sinal = stats.binomtest(neg, len(x), 0.5).pvalue
    print(f'  {nome:10s}  media={x.mean()*100:+6.2f}%   t={t:+5.2f} (p={p_t:.3f})'
          f'   negativos={neg}/{len(x)} (sign p={p_sinal:.3f})')

## 7) A curva CAAR (o grafico classico de event study)

Mediamos o retorno anormal de cada dia relativo entre os 15 eventos e acumulamos. Mostra **quando** a reacao acontece.

In [ ]:
caar = ar_mat.mean(axis=1).cumsum() * 100  # AR medio acumulado, em %
plt.figure()
plt.plot(caar.index, caar.values, marker='o', color='#2a78d6')
plt.axvline(0, color='#e34948', linestyle='--', label='divulgacao (dia 0)')
plt.axhline(0, color='#444', linewidth=0.8)
plt.xlabel('pregao relativo a divulgacao')
plt.ylabel('CAAR (%)')
plt.title('Retorno anormal acumulado medio — Itau')
plt.legend(); plt.tight_layout(); plt.show()

## 8) Leitura

- Em **12 dos 15 eventos** o Itau caiu mais do que o mercado na janela [0,+1]; reacao media **~-0,57%**.
- O **teste do sinal (p~0,035)** diz que 12/15 quedas e improvavel por acaso. O teste t (p~0,056) e marginal, mas aponta junto.
- A curva CAAR mostra a reacao **concentrada nos dias 0 a +2** — coerente com a divulgacao no meio do pregao.

**Conclusao da PoC:** ha reacao negativa do preco do Itau a divulgacao do ranking. O metodo funciona e o sinal move o preco.

**Limitacoes:** so 15 eventos e 1 banco; Itau e peca grande do Ibovespa (modelo de mercado parcialmente mecanico). Isto prova o metodo, nao a tese central (severidade vs. volume) — proximo passo: rodar os 8 bancos e fazer o ablation.

---
### Bonus: trocar de banco
Mude o ticker abaixo para rodar outro banco (ex.: `BBDC4.SA`, `BBAS3.SA`, `SANB11.SA`, `BPAC11.SA`).

In [ ]:
def rodar_banco(ticker):
    p = yf.download([ticker, '^BVSP'], start='2021-06-01', auto_adjust=True, progress=False)['Close'].dropna()
    r = p.pct_change().dropna()
    r.columns = ['r_acao', 'r_mkt'] if r.columns[0] == ticker else ['r_mkt', 'r_acao']
    c, _ = event_study(r, DATAS_DIVULGACAO)
    x = c['CAR(0,1)'].values
    t, p_t = stats.ttest_1samp(x, 0.0)
    neg = int((x < 0).sum())
    print(f'{ticker}: CAR(0,1) medio = {x.mean()*100:+.2f}%   negativos = {neg}/{len(x)}'
          f'   t={t:+.2f} (p={p_t:.3f})')

rodar_banco('ITUB4.SA')